In [1]:
!pip install tensorflow

## Import necessary libraries


In [2]:

import os
import pandas as pd
import numpy as np
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from transformers import BertTokenizer, BertModel, AdamW, BertTokenizerFast, BertForTokenClassification, AutoTokenizer
from sklearn.preprocessing import LabelEncoder

from tqdm import tqdm

## Installing necessary libraries
import tensorflow as tf
import pandas as pd
import numpy as np


import matplotlib.pyplot as plt
import seaborn as sns

import re
import nltk
import spacy
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

import gensim
from gensim.models import Word2Vec, Doc2Vec
from transformers import BertTokenizer, BertModel
import torch
from gensim.models.doc2vec import TaggedDocument
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint


from tensorflow.keras import layers, models
from tensorflow.keras.layers import Bidirectional
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report


import time
import unicodedata



/opt/anaconda3/envs/Conda_3_12_7/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
!python -m spacy download en_core_web_md

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
nlp = spacy.load('en_core_web_md')
pd.set_option('display.max_colwidth',100)


tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased")


  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_md-3.8.0/en_core_web_md-3.8.0-py3-none-any.whl (33.5 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')


[nltk_data] Downloading package punkt to /Users/monilshah/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/monilshah/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/monilshah/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## Getting the data

In [5]:
training_data_path = '/Users/monilshah/Documents/02_NWU/10_MSDS_453_NLP/98_project_work/02_wip_data/relations_train_data.json'
test_data_path = '/Users/monilshah/Documents/02_NWU/10_MSDS_453_NLP/98_project_work/01_Input/02_wip_data/relations_test_data.json'

In [6]:
# Load the JSON file
with open(training_data_path, 'r') as file:
    all_data = json.load(file)

# Example structure of each data entry
# {
#     "news_line": "Sentence text here.",
#     "triples": [
#         {"subject": "Entity1", "object": "Entity2", "relation": "RelationType"}
#     ]
# }



In [7]:
data = all_data[0]

In [8]:
data[0]

{'news_line': 'NEW YORK (Reuters) - Apple Inc Chief Executive Steve Jobs sought to soothe investor concerns about his health on Monday, saying his weight loss was caused by a hormone imbalance that is relatively simple to treat.',
 'triples': [{'subject': 'Apple Inc',
   'object': 'Steve Jobs',
   'relation': 'founded_by'},
  {'subject': 'Apple Inc',
   'object': 'Steve Jobs',
   'relation': 'chief_executive_officer'}]}

In [9]:
# Expand each triple into a row
rows = []
for item in data:  # Loop through each item in the list
    for triple in item['triples']:
        rows.append({
            'news_line': item['news_line'],
            'subject': triple['subject'],
            'object': triple['object'],
            'relation': triple['relation']
        })

# Create the DataFrame
df = pd.DataFrame(rows)
print(df)


                                                                                                news_line  \
0     NEW YORK (Reuters) - Apple Inc Chief Executive Steve Jobs sought to soothe investor concerns abo...   
1     NEW YORK (Reuters) - Apple Inc Chief Executive Steve Jobs sought to soothe investor concerns abo...   
2     Last week, Citigroup Inc's ( C.N ) Chief Executive Vikram Pandit said that he, Chairman Win Bisc...   
3     Lehman Brothers LEH.N shares fell sharply on Monday on speculation that the investment bank coul...   
4     Lehman Brothers LEH.N shares fell sharply on Monday on speculation that the investment bank coul...   
...                                                                                                   ...   
8074  In particular, he said leases of used A330-200 aircraft from Airbus Group SE (Xetra: A1XBMK - ne...   
8075  The company is an omnipresent in households the world over thanks to its operations across multi...   
8076               

In [10]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['relation'] = le.fit_transform(df['relation'])

In [11]:
def get_filtered_entities(text):
    
    text = text.replace("Inc ", "Inc. ")
    
    doc = nlp(text)
    # Collect only entities that are either a person or an organization
    entities = set([ent.text for ent in doc.ents if ent.label_ in {"PERSON", "ORG"}])
    return entities

# Apply the function and convert the set to a comma-separated string
df['entities'] = df['news_line'].apply(lambda x: ', '.join(get_filtered_entities(x)))

print(df)


                                                                                                news_line  \
0     NEW YORK (Reuters) - Apple Inc Chief Executive Steve Jobs sought to soothe investor concerns abo...   
1     NEW YORK (Reuters) - Apple Inc Chief Executive Steve Jobs sought to soothe investor concerns abo...   
2     Last week, Citigroup Inc's ( C.N ) Chief Executive Vikram Pandit said that he, Chairman Win Bisc...   
3     Lehman Brothers LEH.N shares fell sharply on Monday on speculation that the investment bank coul...   
4     Lehman Brothers LEH.N shares fell sharply on Monday on speculation that the investment bank coul...   
...                                                                                                   ...   
8074  In particular, he said leases of used A330-200 aircraft from Airbus Group SE (Xetra: A1XBMK - ne...   
8075  The company is an omnipresent in households the world over thanks to its operations across multi...   
8076               

In [29]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, TimeDistributed, Bidirectional

# Step 1: Tokenize sentences and create labels for each token
def prepare_data(df, maxlen = None):
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(df['news_line'])
    
    X = []
    y = []
    for _, row in df.iterrows():
        tokens = row['news_line'].split()
        labels = [1 if " ".join(tokens[i:i+len(row['subject'].split())]) == row['subject'] else 0 for i in range(len(tokens))]
        
        
        # Handle multi-word subjects by labeling each word in the subject with 1
        for i in range(len(tokens) - len(row['subject'].split()) + 1):
            if " ".join(tokens[i:i + len(row['subject'].split())]) == row['subject']:
                labels[i:i + len(row['subject'].split())] = [1] * len(row['subject'].split())

        # Append tokenized text and labels
        X.append(tokens)
        y.append(labels)
    
    # Convert texts to sequences of integers
    X_seq = tokenizer.texts_to_sequences([" ".join(tokens) for tokens in X])
    
    if maxlen is None:
        maxlen = max(len(seq) for seq in X_seq)
    
    # Pad sequences and labels
    X_pad = pad_sequences(X_seq, padding='post', maxlen= maxlen)
    y_pad = pad_sequences(y, padding='post', maxlen= maxlen)

    return X_pad, np.array(y_pad), tokenizer



In [30]:
# Step 2: Split data into train, validation, and test sets
# First, split out the test data (20% test)
df_train_val, df_test = train_test_split(df, test_size=0.2, random_state=42)

# Now, split the remaining data into train and validation sets (80% train, 20% validation)
X, y, tokenizer = prepare_data(df_train_val)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)




In [31]:
X.shape, y.shape

((6463, 998), (6463, 998))

In [32]:
# Step 3: Define LSTM model for sequence tagging
model = Sequential([
    Embedding(input_dim=len(tokenizer.word_index) + 1, output_dim=64, input_length=X.shape[1]),
    Bidirectional(LSTM(64, return_sequences=True)),
    TimeDistributed(Dense(1, activation='sigmoid'))
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Step 4: Train the model
model.fit(X_train, y_train, epochs=5, batch_size=1, validation_data=(X_val, y_val))

# Step 5: Evaluate the model on test data
X_test, y_test, _ = prepare_data(df_test)  # Prepare the test data
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy:.2f}")

Epoch 1/5


/opt/anaconda3/envs/Conda_3_12_7/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


5170/5170 ━━━━━━━━━━━━━━━━━━━━ 1363s 261ms/step - accuracy: 0.9969 - loss: 0.0102 - val_accuracy: 0.9987 - val_loss: 0.0042
Epoch 2/5
5170/5170 ━━━━━━━━━━━━━━━━━━━━ 1263s 244ms/step - accuracy: 0.9989 - loss: 0.0033 - val_accuracy: 0.9988 - val_loss: 0.0039
Epoch 3/5
5170/5170 ━━━━━━━━━━━━━━━━━━━━ 1253s 242ms/step - accuracy: 0.9991 - loss: 0.0025 - val_accuracy: 0.9988 - val_loss: 0.0040
Epoch 4/5
5170/5170 ━━━━━━━━━━━━━━━━━━━━ 1320s 255ms/step - accuracy: 0.9993 - loss: 0.0019 - val_accuracy: 0.9987 - val_loss: 0.0043
Epoch 5/5
5170/5170 ━━━━━━━━━━━━━━━━━━━━ 1268s 245ms/step - accuracy: 0.9994 - loss: 0.0016 - val_accuracy: 0.9987 - val_loss: 0.0045
51/51 ━━━━━━━━━━━━━━━━━━━━ 6s 56ms/step - accuracy: 0.9983 - loss: 0.0105
Test Accuracy: 1.00
